## Step 2  
Input: s3://thesis--ec331-s3/enriched-volume-bids/  
Output: s3://thesis--ec331-s3/melted-volume-bids/  

TODO: 
- I need to recognise what has been loaded and not to avoid duplication
- Before sending to s3, remove all zero rows!


In [1]:
# Melt and remove zero rows at the same time
def melt_and_write_chunks(df, chunk_size=50_000, output_folder="", file_label=""):
    """
    Melts the BANDAVAIL columns in chunks and writes each chunk directly to S3.
    Filenames: {output_folder}/{file_label}_partXXXX.parquet

    Returns: (list_of_chunk_paths, total_rows_processed)
    """
    print(f"Starting streaming melt of DataFrame with shape: {df.shape}")
    print(f"Current memory usage: {get_memory_usage():.2f} GB")

    start_time = time.time()
    num_rows = len(df)
    num_chunks = (num_rows + chunk_size - 1) // chunk_size
    print(f"Processing in {num_chunks} chunks of size {chunk_size}")

    total_rows_processed = 0
    chunk_paths = []

    for i in range(num_chunks):
        chunk_start = i * chunk_size
        chunk_end = min((i + 1) * chunk_size, num_rows)

        print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
        print(f"Memory before chunk: {get_memory_usage():.2f} GB")

        # Copy just this chunk
        chunk = df.iloc[chunk_start:chunk_end].copy()

        # Identify columns not in BAND_COLS
        id_vars_cols = [col for col in chunk.columns if col not in BAND_COLS]

        # Melt the BANDAVAIL columns
        chunk_melted = pd.melt(
            chunk,
            id_vars=id_vars_cols,
            value_vars=[c for c in BAND_COLS if c in chunk.columns],
            var_name="BIDBAND",
            value_name="BIDVOLUME"
        )

        del chunk
        gc.collect()

        # Extract numeric band from e.g. "BANDAVAIL3"
        chunk_melted["BIDBAND"] = (
            chunk_melted["BIDBAND"]
            .str.extract(r"BANDAVAIL(\d+)")
            .astype(int)
        )

        # Optional type transformations
        if "BIDTYPE" in chunk_melted.columns:
            chunk_melted["BIDTYPE"] = chunk_melted["BIDTYPE"].astype(str)
        if "DUID" in chunk_melted.columns:
            chunk_melted["DUID"] = chunk_melted["DUID"].astype(str)
        if "SETTLEMENTDATE" in chunk_melted.columns:
            chunk_melted["SETTLEMENTDATE"] = pd.to_datetime(chunk_melted["SETTLEMENTDATE"])

        # Drop rows where BIDVOLUME is null
        chunk_melted.dropna(subset=["BIDVOLUME"], inplace=True)
        
        # NEW CODE: Remove rows where BIDVOLUME = 0
        initial_rows = len(chunk_melted)
        chunk_melted = chunk_melted[chunk_melted['BIDVOLUME'] != 0]
        removed_rows = initial_rows - len(chunk_melted)
        print(f"  Removed {removed_rows} rows where BIDVOLUME = 0 ({(removed_rows/initial_rows)*100:.2f}% of chunk)")

        chunk_len = len(chunk_melted)
        total_rows_processed += chunk_len

        # Construct output filename
        chunk_output = f"{output_folder.rstrip('/')}/{file_label}_part{i+1:04d}.parquet"
        try:
            wr.s3.to_parquet(
                df=chunk_melted,
                path=chunk_output,
                index=False,
                compression="snappy"
            )
            chunk_paths.append(chunk_output)
            print(f"  ✓ Wrote chunk {i+1}/{num_chunks} with {chunk_len} rows to {chunk_output}")
        except Exception as e:
            print(f"  ✗ Error writing chunk {i+1}: {str(e)}")

        del chunk_melted
        gc.collect()
        print(f"Memory usage after chunk {i+1}: {get_memory_usage():.2f} GB")

    total_time = time.time() - start_time
    print(f"\nAll chunks processed in {total_time:.2f} seconds.")
    print(f"Total rows processed: {total_rows_processed}")
    print(f"Final memory usage: {get_memory_usage():.2f} GB\n")

    return chunk_paths, total_rows_processed

PROCESSING VOLUME BIDS (MELT BID VOLUME)
INPUT: s3://thesis--ec331-s3/enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/<parquet-files>
OUTPUT: s3://thesis--ec331-s3/melted-volume-bids/<matching-subfolders>/<chunks>


Found 5 .parquet file(s) under enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/.

Starting to process: s3://thesis--ec331-s3/enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/enriched_volume_bids_20250320_085431_chunk1of5.parquet

Processing single file: s3://thesis--ec331-s3/enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/enriched_volume_bids_20250320_085431_chunk1of5.parquet
Output subfolder: s3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_capped/
File label: enriched_volume_bids_20250320_085431_chunk1of5
Read file in 0.27 seconds. Shape: (10000, 47)
Memory usage after reading: 0.19 GB
Starting streaming melt of DataFrame with s

In [4]:
import awswrangler as wr
files = wr.s3.list_objects("s3://thesis--ec331-s3/enriched-volume-bids/")
print(f"Found {len(files)} files")

Found 2123 files
